# Combine automatic metrics (WER, UTMOSv2) with human MOS

Builds a single long-format dataframe with one row per (sample, annotator):

`filename, text, audio_path, dataset, model, language, wer, utmosv2, annotator, mos`

- **Automatic metrics** come from `data/audios/{dataset}/{model}/{language}/results.csv` (`tts_wer`, `tts_utmos`).
- **Human MOS** comes from `data/annotations-anonymized/{Language}.csv` (3 annotators per sample).

The two sources are joined on `(dataset, model, language, filename)`, which is parsed out of each annotation's `audio_url`.

In [12]:
import glob
import os
import urllib.parse

import pandas as pd

AUDIOS_DIR = "data/audios"
ANNOTATIONS_DIR = "data/annotations-anonymized"

In [13]:
# ── 1. Automatic metrics ─────────────────────────────────────────────
# data/audios/{dataset}/{model}/{language}/results.csv
auto_frames = []
for path in glob.glob(os.path.join(AUDIOS_DIR, "*", "*", "*", "results.csv")):
    dataset, model, language = path.split(os.sep)[-4:-1]
    df = pd.read_csv(path)
    df["dataset"] = dataset
    df["model"] = model
    df["language"] = language
    auto_frames.append(df)

auto = pd.concat(auto_frames, ignore_index=True)
auto = auto.rename(columns={"tts_wer": "wer", "tts_utmos": "utmosv2"})
auto = auto[["dataset", "model", "language", "filename", "text", "wer", "utmosv2"]]
print(f"Automatic metrics: {len(auto)} samples")
auto.head()

Automatic metrics: 3380 samples


,dataset,model,language,filename,text,wer,utmosv2
0,open-bible,F5,Haitian Creole,1KI_020_Verse_042.wav,Li te di li: “Konsa pale SENYÈ a: ‘Akoz ou te ...,0.534884,2.748047
1,open-bible,F5,Haitian Creole,EZR_010_Verse_017.wav,Yo te fin fè rechèch la sou tout mesye ki te m...,0.375000,2.820312
2,open-bible,F5,Haitian Creole,EZR_005_Verse_009.wav,"Konsa, nou te kesyonen ansyen sila yo epi te m...",0.896552,2.789062
3,open-bible,F5,Haitian Creole,PSA_018_Verse_036.wav,"Ou fè longè pa m yo lonje pi laj anba m, epi p...",0.277778,2.439453
4,open-bible,F5,Haitian Creole,ISA_060_Verse_003.wav,"Nasyon yo va vin kote limyè ou a, e wa yo nan ...",0.705882,3.173828


In [14]:
# ── 2. Human annotations ─────────────────────────────────────────────
# One row per (sample, annotator); 3 annotators per sample.
anno = pd.concat(
    [pd.read_csv(p) for p in glob.glob(os.path.join(ANNOTATIONS_DIR, "*.csv"))],
    ignore_index=True,
)

# Some samples were rated more than once -> keep only the most recent rating
# for each (annotator, audio_url).
anno["updated_at"] = pd.to_datetime(anno["updated_at"])
n_before = len(anno)
anno = (
    anno.sort_values("updated_at", ascending=False)
    .drop_duplicates(subset=["annotator", "audio_url"], keep="first")
    .sort_index()
    .reset_index(drop=True)
)
print(f"Deduplicated annotations: {n_before} -> {len(anno)} rows")

# dataset/model/language/filename are encoded in the audio_url:
#   .../audios/{dataset}/{model}/{language}/{filename}
url_parts = anno["audio_url"].str.split("/audios/").str[1].str.split("/")
anno["model"] = url_parts.str[1]
anno["language"] = url_parts.str[2].map(urllib.parse.unquote)  # "Haitian%20Creole" -> "Haitian Creole"
# anno already has `dataset` and `filename` columns matching the url.

# mos is stored as e.g. "3 — Fair" -> 3
anno["mos"] = anno["mos"].str.split().str[0].astype(int)

anno = anno[["dataset", "model", "language", "filename", "annotator", "mos"]]
print(f"Human annotations: {len(anno)} rows "
      f"({anno.groupby(['dataset', 'model', 'language', 'filename']).ngroups} unique samples)")
anno.head()

Deduplicated annotations: 10146 -> 10140 rows
Human annotations: 10140 rows (3380 unique samples)


,dataset,model,language,filename,annotator,mos
0,bouquet,OmniVoice,Haitian Creole,P187-S5.wav,annotator_1,2
1,bouquet,OmniVoice,Haitian Creole,P187-S5.wav,annotator_2,2
2,bouquet,OmniVoice,Haitian Creole,P187-S5.wav,annotator_3,3
3,fleurs,EveryVoice,Haitian Creole,520.wav,annotator_1,2
4,fleurs,EveryVoice,Haitian Creole,520.wav,annotator_2,2


In [16]:
# ── 3. Join automatic + human ────────────────────────────────────────
key = ["dataset", "model", "language", "filename"]
data = anno.merge(auto, on=key, how="left")

# Local path to the audio file
data["audio_path"] = data.apply(
    lambda r: os.path.join(AUDIOS_DIR, r["dataset"], r["model"], r["language"], r["filename"]),
    axis=1,
)

data = data[
    ["filename", "text", "audio_path", "dataset", "model",
     "language", "wer", "utmosv2", "annotator", "mos"]
]

# Sanity check: every annotation should have matched an automatic-metrics row.
missing = data["utmosv2"].isna().sum()
assert missing == 0, f"{missing} rows did not match any results.csv"
print(f"Combined dataframe: {len(data)} rows")
data.head()

Combined dataframe: 10140 rows


,filename,text,audio_path,dataset,model,language,wer,utmosv2,annotator,mos
0,P187-S5.wav,"Si pa aza gen yon bagay ki ta rive m, pwomèt m...",data/audios/bouquet/OmniVoice/Haitian Creole/P...,bouquet,OmniVoice,Haitian Creole,0.205882,3.230469,annotator_1,2
1,P187-S5.wav,"Si pa aza gen yon bagay ki ta rive m, pwomèt m...",data/audios/bouquet/OmniVoice/Haitian Creole/P...,bouquet,OmniVoice,Haitian Creole,0.205882,3.230469,annotator_2,2
2,P187-S5.wav,"Si pa aza gen yon bagay ki ta rive m, pwomèt m...",data/audios/bouquet/OmniVoice/Haitian Creole/P...,bouquet,OmniVoice,Haitian Creole,0.205882,3.230469,annotator_3,3
3,520.wav,"Pa egzanp, “aprantisaj” and “sosyalizasyon” ko...",data/audios/fleurs/EveryVoice/Haitian Creole/5...,fleurs,EveryVoice,Haitian Creole,1.062500,2.824219,annotator_1,2
4,520.wav,"Pa egzanp, “aprantisaj” and “sosyalizasyon” ko...",data/audios/fleurs/EveryVoice/Haitian Creole/5...,fleurs,EveryVoice,Haitian Creole,1.062500,2.824219,annotator_2,2


In [ ]:
# ── 4. Spearman correlation (utmosv2 vs mos) per language ─────────────
from scipy.stats import spearmanr

spearman_per_language = (
    data.groupby("language")
    .apply(lambda g: pd.Series({
        "spearman": spearmanr(g["utmosv2"], g["mos"]).correlation,
        "p_value": spearmanr(g["utmosv2"], g["mos"]).pvalue,
        "n": len(g),
    }))
    .sort_values("spearman", ascending=False)
)
spearman_per_language

In [ ]:
# ── 5. Compare: raw (3 annotators repeated) vs per-sample mean MOS ────
# Per-sample mean: average the 3 annotator ratings, one row per utterance.
per_sample = (
    data.groupby(["language", "dataset", "model", "filename"], as_index=False)
    .agg(utmosv2=("utmosv2", "first"), mos=("mos", "mean"))
)

def spearman_by_language(df):
    return df.groupby("language").apply(
        lambda g: pd.Series({
            "spearman": spearmanr(g["utmosv2"], g["mos"]).correlation,
            "p_value": spearmanr(g["utmosv2"], g["mos"]).pvalue,
            "n": len(g),
        })
    )

raw = spearman_by_language(data)        # 3 annotator rows per sample
mean = spearman_by_language(per_sample)  # one mean-MOS row per sample

comparison = pd.DataFrame({
    "spearman_raw": raw["spearman"],
    "spearman_mean": mean["spearman"],
    "n_raw": raw["n"].astype(int),
    "n_mean": mean["n"].astype(int),
}).sort_values("spearman_mean", ascending=False)
comparison